# 01 — Dataset Collection & Preprocessing
**AnemiaFusionNet — Stage 1**

This notebook loads the clinical CBC dataset and image patient datasets, maps raw image paths to patients, performs label engineering using WHO guidelines, simulates clinical vectors and geographical states, and splits the data into patient-level train/val/test partitions.

## 1.1 Set Working Directory and Verify Datasets


In [37]:
import os
from pathlib import Path

# Change working directory to project root if executed from notebooks folder
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
print("Current working directory:", os.getcwd())

RAW_IMG_DIR = Path("data/raw/images/dataset anemia")
RAW_CLINICAL = Path("data/raw/clinical/diagnosed_cbc_data_v4.csv")




Current working directory: c:\Users\ratha\OneDrive\Desktop\datavidwan\New folder (2)\files


## 1.2 Load Raw Spreadsheets and Clean Hgb / Age Columns

In [38]:
import pandas as pd
import numpy as np

# Load Clinical CBC dataset
clinical_df = pd.read_csv(RAW_CLINICAL)
clinical_df.shape



(1281, 15)

In [39]:
clinical_df.head()

,WBC,LYMp,NEUTp,LYMn,NEUTn,RBC,HGB,HCT,MCV,MCH,MCHC,PLT,PDW,PCT,Diagnosis
0,10.0,43.2,50.1,4.3,5.0,2.77,7.3,24.2,87.7,26.3,30.1,189.0,12.5,0.17,Normocytic hypochromic anemia
1,10.0,42.4,52.3,4.2,5.3,2.84,7.3,25.0,88.2,25.7,20.2,180.0,12.5,0.16,Normocytic hypochromic anemia
2,7.2,30.7,60.7,2.2,4.4,3.97,9.0,30.5,77.0,22.6,29.5,148.0,14.3,0.14,Iron deficiency anemia
3,6.0,30.2,63.5,1.8,3.8,4.22,3.8,32.8,77.9,23.2,29.8,143.0,11.3,0.12,Iron deficiency anemia
4,4.2,39.1,53.7,1.6,2.3,3.93,0.4,316.0,80.6,23.9,29.7,236.0,12.8,0.22,Normocytic hypochromic anemia


In [40]:
# Map Diagnosis categories to binary labels: Any diagnosis with 'anemia' -> 1, others -> 0
clinical_df["label"] = clinical_df["Diagnosis"].apply(lambda x: 1 if "anemia" in str(x).lower() else 0)
print("Clinical label distribution:\n", clinical_df["label"].value_counts())



Clinical label distribution:
 label
1    814
0    467
Name: count, dtype: int64


In [41]:
# Save clinical labels to csv for standalone pretraining in notebook 04
os.makedirs("data/processed", exist_ok=True)
clinical_df.to_csv("data/processed/clinical_pretrain.csv", index=False)


In [42]:

# Load Image spreadsheets
india_meta = pd.read_excel(RAW_IMG_DIR / "India/India.xlsx")
italy_meta = pd.read_excel(RAW_IMG_DIR / "Italy/Italy.xlsx")


In [43]:
india_meta.shape


(95, 5)

In [44]:
india_meta.head()

,Number,Hgb,Gender,Age,Note
0,1,12.2,M,29,NaN
1,2,8.0,F,36,NaN
2,3,10.7,F,30,NaN
3,4,8.3,F,39,NaN
4,5,7.8,F,29,NaN


In [45]:

# Clean and standardize columns
india_meta = india_meta[["Number", "Hgb", "Gender", "Age"]].copy()
india_meta["dataset"] = "India"

italy_meta = italy_meta[["Number", "Hgb", "Gender", "Age"]].copy()
italy_meta["dataset"] = "Italy"

img_meta = pd.concat([india_meta, italy_meta], ignore_index=True)
print("Raw combined image meta shape:", img_meta.shape)



Raw combined image meta shape: (218, 5)


In [46]:
img_meta.tail(100)

,Number,Hgb,Gender,Age,dataset
118,24,9.2,M,56,Italy
119,25,8.7,M,53,Italy
120,26,8.6,M,60,Italy
121,27,9.35,M,59,Italy
122,28,15.45,M,22,Italy
...,...,...,...,...,...
213,119,16.7,M,41,Italy
214,120,15.4,M,22,Italy
215,121,12.8,F,40,Italy
216,122,15.7,M,33,Italy


In [47]:
# Define function to parse Hgb and Age strings
def clean_hgb(val):
    if pd.isna(val):
        return np.nan
    val_str = str(val).strip().replace(",", ".")
    if val_str == "_" or val_str == "":
        return np.nan
    try:
        return float(val_str)
    except:
        return np.nan



In [48]:
img_meta["Hgb"] = img_meta["Hgb"].apply(clean_hgb)
img_meta["Age"] = pd.to_numeric(img_meta["Age"], errors="coerce")

# Drop rows where Hgb is missing
img_meta = img_meta.dropna(subset=["Hgb"]).reset_index(drop=True)
print("Cleaned combined image meta shape:", img_meta.shape)
img_meta.head()

Cleaned combined image meta shape: (217, 5)


,Number,Hgb,Gender,Age,dataset
0,1,12.2,M,29,India
1,2,8.0,F,36,India
2,3,10.7,F,30,India
3,4,8.3,F,39,India
4,5,7.8,F,29,India


## 1.3 Map Patient to Image Path
Write a function to locate the `.jpg` or `.jpeg` raw eye photo inside each patient folder.

In [49]:
def get_raw_image_path(row):
    dataset = row["dataset"]
    num = int(row["Number"])
    pat_dir = RAW_IMG_DIR / dataset / str(num)
    
    # Locate .jpg/.jpeg files in folder
    jpgs = list(pat_dir.glob("*.jpg")) + list(pat_dir.glob("*.jpeg"))
    if len(jpgs) == 1:
        # Return path relative to the project root
        return f"data/raw/images/dataset anemia/{dataset}/{num}/{jpgs[0].name}"
    else:
        raise ValueError(f"Could not map raw image for {dataset}/{num}: found {len(jpgs)} files")

img_meta["image_path"] = img_meta.apply(get_raw_image_path, axis=1)
print(img_meta[["Number", "dataset", "image_path"]].head())

   Number dataset                                         image_path
0       1   India  data/raw/images/dataset anemia/India/1/2020011...
1       2   India  data/raw/images/dataset anemia/India/2/2020012...
2       3   India  data/raw/images/dataset anemia/India/3/2020012...
3       4   India  data/raw/images/dataset anemia/India/4/2020012...
4       5   India  data/raw/images/dataset anemia/India/5/2020012...


## 1.4 Label Engineering using WHO Thresholds
Apply WHO guidelines: Anemic if Hgb < 11.0 g/dL (children < 15y), Hgb < 12.0 g/dL (women), Hgb < 13.0 g/dL (men).

In [50]:
def who_label(hb, sex, age):
    if pd.isna(age) or age < 15:
        threshold = 11.0
    elif str(sex).upper().startswith("F"):
        threshold = 12.0
    else:
        threshold = 13.0
    return int(hb < threshold)

img_meta["label"] = img_meta.apply(lambda r: who_label(r["Hgb"], r["Gender"], r["Age"]), axis=1)
print("WHO Label distribution for image patients:\n", img_meta["label"].value_counts())

WHO Label distribution for image patients:
 label
0    126
1     91
Name: count, dtype: int64


In [51]:
img_meta.head()

,Number,Hgb,Gender,Age,dataset,image_path,label
0,1,12.2,M,29,India,data/raw/images/dataset anemia/India/1/2020011...,1
1,2,8.0,F,36,India,data/raw/images/dataset anemia/India/2/2020012...,1
2,3,10.7,F,30,India,data/raw/images/dataset anemia/India/3/2020012...,1
3,4,8.3,F,39,India,data/raw/images/dataset anemia/India/4/2020012...,1
4,5,7.8,F,29,India,data/raw/images/dataset anemia/India/5/2020012...,1


## 1.5 Simulate Modality Linkage & Geographical Prior
Since no public dataset links the same patient's eye photo, clinical CBC, and geographic prior, we:
1. Simulate geographic states for India and Italy patients (using Indian state weights).
2. Sample clinical features from the clinical CBC dataset matching the patient's anemic/healthy status to simulate joint linkage.

In [52]:
INDIAN_STATES_SAMPLE_WEIGHTS = {
    "Uttar Pradesh": 0.166, "Maharashtra": 0.093, "Bihar": 0.086, "West Bengal": 0.076,
    "Madhya Pradesh": 0.060, "Tamil Nadu": 0.058, "Rajasthan": 0.056, "Karnataka": 0.050,
    "Gujarat": 0.050, "Andhra Pradesh": 0.041, "Odisha": 0.035, "Telangana": 0.029,
    "Kerala": 0.027, "Assam": 0.026, "Punjab": 0.023, "Chhattisgarh": 0.021,
    "Haryana": 0.021, "Jharkhand": 0.027, "Delhi": 0.014,
}

states, weights = zip(*INDIAN_STATES_SAMPLE_WEIGHTS.items())
weights = np.array(weights) / np.sum(weights)

np.random.seed(42)
img_meta["state"] = np.random.choice(states, size=len(img_meta), p=weights)
img_meta["geo_source"] = "simulated"



In [55]:
# Simulate Clinical features based on target label
clinical_cols = [c for c in clinical_df.columns if c not in ["Diagnosis", "label"]]
simulated_clinical_list = []

for idx, row in img_meta.iterrows():
    label = row["label"]
    # Filter clinical rows matching the same target class (Anemic vs. Healthy)
    matching_rows = clinical_df[clinical_df["label"] == label]
    sampled_row = matching_rows.sample(n=1, random_state=idx).iloc[0]
    
    # Extract numeric CBC features
    clin_feats = sampled_row[clinical_cols].to_dict()
    
    
    clin_feats["region_type"] = np.random.choice(["urban", "rural"], p=[0.35, 0.65])
    
    simulated_clinical_list.append(clin_feats)

sim_clinical_df = pd.DataFrame(simulated_clinical_list)
master_df = pd.concat([img_meta, sim_clinical_df], axis=1)
master_df = master_df.drop(columns=["HGB"], errors="ignore")
print("Master dataset columns:", master_df.columns.tolist())
print("Master dataset shape:", master_df.shape)
master_df.to_csv("data/processed/master_dataset.csv", index=False)

Master dataset columns: ['Number', 'Hgb', 'Gender', 'Age', 'dataset', 'image_path', 'label', 'state', 'geo_source', 'WBC', 'LYMp', 'NEUTp', 'LYMn', 'NEUTn', 'RBC', 'HCT', 'MCV', 'MCH', 'MCHC', 'PLT', 'PDW', 'PCT', 'region_type']
Master dataset shape: (217, 23)


## 1.6 Patient-level Stratified Partitioning
Partition the dataset into Train (70%), Val (15%), and Test (15%) splits, ensuring stratification by label.

In [56]:
from sklearn.model_selection import train_test_split

train_df, val_test_df = train_test_split(
    master_df, 
    test_size=0.30, 
    stratify=master_df["label"], 
    random_state=42
)

val_df, test_df = train_test_split(
    val_test_df, 
    test_size=0.50, 
    stratify=val_test_df["label"], 
    random_state=42
)

print(f"Train size: {len(train_df)} | Val size: {len(val_df)} | Test size: {len(test_df)}")
train_df.to_csv("data/processed/train.csv", index=False)
val_df.to_csv("data/processed/val.csv", index=False)
test_df.to_csv("data/processed/test.csv", index=False)
print("Splits successfully saved to data/processed/")

Train size: 151 | Val size: 33 | Test size: 33
Splits successfully saved to data/processed/
